# 07 — DP-CTGAN: GAN Tabular con Differential Privacy

**Fase 2 — Contribución diferencial del TFG**

Este notebook implementa la evaluación del tradeoff privacidad–utilidad usando
**Differential Privacy via DP-SGD** (Abadi et al., 2016) con la librería **Opacus** (Meta AI).

La privacidad diferencial (ε, δ)-DP garantiza que la probabilidad de inferir
si un paciente concreto estuvo en el dataset de entrenamiento está acotada
por un factor e^ε. Cuanto menor ε, mayor privacidad — pero también mayor
degradación de la calidad de los datos sintéticos.

**Experimento**: entrenar DP-CTGAN para ε ∈ {1, 5, 10, ∞} y medir la pérdida
de fidelidad estadística en función del presupuesto de privacidad ε.

Referencias:
- Abadi et al., *Deep Learning with Differential Privacy* (CCS 2016)
- Yousefpour et al., *Opacus: User-Friendly Differential Privacy Library in PyTorch* (2021)

## 0. Imports y configuración

In [ ]:
import sys, subprocess, warnings, time
warnings.filterwarnings("ignore")

try:
    import opacus
    print(f"Opacus {opacus.__version__} ya instalado.")
except ImportError:
    print("Instalando Opacus...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opacus", "-q"])
    import opacus
    print(f"Opacus {opacus.__version__} instalado.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.accountants.utils import get_noise_multiplier

ROOT      = Path("..")
PROCESSED = ROOT / "data" / "processed"
SYNTHETIC = ROOT / "data" / "synthetic"
MODELS    = ROOT / "models"
REPORTS   = ROOT / "reports"
for d in [SYNTHETIC, MODELS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))
from models.dp_ctgan import TabGANGenerator, TabGANDiscriminator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

## 1. Carga y preprocesamiento

In [ ]:
# Mismo preprocesador que en notebook 05 (TabDDPM)
class TabularPreprocessor:
    def __init__(self, num_cols, binary_cols, cat_cols):
        self.num_cols    = num_cols
        self.binary_cols = binary_cols
        self.cat_cols    = cat_cols
        self.input_dim   = len(num_cols) + len(binary_cols) + len(cat_cols)
        self.num_scaler  = StandardScaler() if num_cols else None
        self.cat_encoder = (
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            if cat_cols else None
        )

    def fit(self, df):
        if self.num_scaler:
            self.num_scaler.fit(df[self.num_cols])
        if self.cat_encoder:
            self.cat_encoder.fit(df[self.cat_cols])
        return self

    def transform(self, df):
        parts = []
        if self.num_scaler:
            parts.append(self.num_scaler.transform(df[self.num_cols]).astype(np.float32))
        if self.binary_cols:
            parts.append(df[self.binary_cols].values.astype(np.float32) * 2 - 1)
        if self.cat_encoder:
            enc = self.cat_encoder.transform(df[self.cat_cols]).astype(np.float32)
            for i in range(enc.shape[1]):
                n = len(self.cat_encoder.categories_[i])
                enc[:, i] = enc[:, i] / max(n - 1, 1) * 2 - 1
            parts.append(enc)
        return np.concatenate(parts, axis=1)

    def fit_transform(self, df):
        return self.fit(df).transform(df)

    def inverse_transform(self, arr):
        result = {}
        idx = 0
        if self.num_scaler:
            n   = len(self.num_cols)
            inv = self.num_scaler.inverse_transform(arr[:, idx:idx + n])
            for i, col in enumerate(self.num_cols):
                result[col] = inv[:, i]
            idx += n
        if self.binary_cols:
            n   = len(self.binary_cols)
            raw = arr[:, idx:idx + n]
            for i, col in enumerate(self.binary_cols):
                result[col] = np.clip(np.round((raw[:, i] + 1) / 2), 0, 1).astype(int)
            idx += n
        if self.cat_encoder:
            n   = len(self.cat_cols)
            enc = arr[:, idx:idx + n].copy()
            for i in range(n):
                n_cats    = len(self.cat_encoder.categories_[i])
                enc[:, i] = np.clip(np.round((enc[:, i] + 1) / 2 * (n_cats - 1)), 0, n_cats - 1)
            inv = self.cat_encoder.inverse_transform(enc.astype(int))
            for i, col in enumerate(self.cat_cols):
                result[col] = inv[:, i]
        return pd.DataFrame(result)


tab = pd.read_parquet(PROCESSED / "tabular_48h.parquet")
TARGET  = "hospital_expire_flag"
id_cols = [c for c in tab.columns if c.endswith("_id")]
feature_cols = [c for c in tab.columns if c not in id_cols + [TARGET]]

# dtype != object evita clasificar columnas string ('M'/'F') como binarias numéricas
binary_cols  = [c for c in feature_cols if tab[c].dropna().nunique() <= 2 and tab[c].dtype != object]
cat_cols     = [c for c in feature_cols if c not in binary_cols and tab[c].dtype == object]
num_cols     = [c for c in feature_cols if c not in binary_cols + cat_cols]

preprocessor = TabularPreprocessor(num_cols, binary_cols, cat_cols)
X = preprocessor.fit_transform(tab)
y = tab[TARGET].values.astype(np.int64)

INPUT_DIM = preprocessor.input_dim
print(f"Dataset: {tab.shape}  →  tensor: {X.shape}")
print(f"input_dim: {INPUT_DIM}  |  mortalidad: {y.mean()*100:.1f}%")

## 2. Hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| `hidden_dims` | (256, 256) | Igual que CTGAN (notebook 04) para comparabilidad directa |
| `noise_dim` | 128 | Igual que `embedding_dim` de CTGAN |
| `batch_size` | 1024 | Batches más grandes mejoran la relación señal-ruido del gradiente privado |
| `N_EPOCHS` | 300 | Mismo que el default de CTGAN; suficiente para ver el efecto del DP |
| `max_grad_norm` | 1.0 | Umbral de clipping del gradiente (C en DP-SGD); valor estándar en la literatura |
| `delta` | 1e-5 | Probabilidad de fallo del mecanismo DP; convención: δ << 1/N |
| ε targets | {1, 5, 10, ∞} | Barrido que cubre desde alta privacidad (ε=1) hasta sin privacidad (ε=∞) |

In [ ]:
NOISE_DIM      = 128
HIDDEN_DIMS    = (256, 256)
BATCH          = 1024
N_EPOCHS       = 300
MAX_GRAD_NORM  = 1.0
DELTA          = 1e-5
LR_G           = 2e-4
LR_D           = 2e-4

# Epsilon targets: None = sin DP (eps=inf)
EPSILON_TARGETS = [None, 10.0, 5.0, 1.0]
EPSILON_LABELS  = ["ε=∞", "ε=10", "ε=5", "ε=1"]

# Columnas clave para medir utilidad
KEY_COLS = [c for c in [
    "heart_rate_mean", "sbp_mean", "spo2_mean", "gcs_total_mean",
    "resp_rate_mean",  "lactate_mean", "creatinine_mean", "age", "los",
] if c in tab.columns]

## 3. Función de entrenamiento

La función `train_dp_gan` encapsula el entrenamiento para un valor de ε dado.
Cuando `target_epsilon=None` entrena sin DP (ε=∞);
en caso contrario usa `PrivacyEngine.make_private_with_epsilon` de Opacus,
que calcula automáticamente el `noise_multiplier` σ necesario para alcanzar
el presupuesto ε con el número de épocas y tamaño de batch especificados.

In [ ]:
def train_dp_gan(
    X_data: np.ndarray,
    target_epsilon: float | None,
    n_epochs: int = N_EPOCHS,
    batch_size: int = BATCH,
    label: str = "",
) -> dict:
    """
    Entrena un GAN tabular con o sin Differential Privacy.

    Para la actualización del generador se usa D._module (módulo interno del
    GradSampleModule de Opacus), que comparte parámetros con D pero no tiene
    los hooks de per-sample gradient. Esto evita que l_g.backward() active
    la contabilidad DP del discriminador.
    """
    G = TabGANGenerator(NOISE_DIM, INPUT_DIM, HIDDEN_DIMS).to(DEVICE)
    D = TabGANDiscriminator(INPUT_DIM, HIDDEN_DIMS).to(DEVICE)

    errors = ModuleValidator.validate(D, strict=False)
    if errors:
        D = ModuleValidator.fix(D)
        print(f"  [Opacus] Discriminador corregido automáticamente ({len(errors)} issue/s).")

    opt_g = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

    X_tensor = torch.from_numpy(X_data).float()
    dataset  = TensorDataset(X_tensor)
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                          drop_last=False, num_workers=0)

    actual_epsilon   = float("inf")
    noise_multiplier = 0.0
    privacy_engine   = None

    if target_epsilon is not None:
        sample_rate = batch_size / len(dataset)
        nm = get_noise_multiplier(
            target_epsilon=target_epsilon,
            target_delta=DELTA,
            sample_rate=sample_rate,
            epochs=n_epochs,
            accountant="rdp",
        )
        privacy_engine = PrivacyEngine()
        D, opt_d, loader = privacy_engine.make_private(
            module=D,
            optimizer=opt_d,
            data_loader=loader,
            noise_multiplier=nm,
            max_grad_norm=MAX_GRAD_NORM,
            poisson_sampling=False,
        )
        noise_multiplier = opt_d.noise_multiplier
        print(f"  [{label}] noise_multiplier σ = {noise_multiplier:.4f}")

    # D_for_G: módulo interno sin hooks de Opacus, parámetros compartidos con D.
    # En el caso sin DP, es el mismo objeto D.
    D_for_G = getattr(D, "_module", D)

    losses_g, losses_d = [], []
    t0 = time.time()

    for epoch in tqdm(range(1, n_epochs + 1), desc=f"Entrenando {label}", leave=False):
        g_ep, d_ep = [], []

        for (x_real,) in loader:
            x_real = x_real.to(DEVICE)
            B      = x_real.size(0)

            # --- Discriminador (con DP-SGD si privacy_engine activo) ---
            z      = torch.randn(B, NOISE_DIM, device=DEVICE)
            x_fake = G(z).detach()
            d_real = D(x_real)
            d_fake = D(x_fake)
            l_d    = (
                F.binary_cross_entropy_with_logits(d_real, torch.ones_like(d_real))
                + F.binary_cross_entropy_with_logits(d_fake, torch.zeros_like(d_fake))
            )
            opt_d.zero_grad()
            l_d.backward()
            opt_d.step()
            d_ep.append(l_d.item())

            # --- Generador ---
            # D_for_G no tiene hooks de Opacus → l_g.backward() no activa
            # per-sample gradient computation en D. Los parámetros son los mismos
            # que los de D (compartidos), así que el gradiente hacia G es correcto.
            z      = torch.randn(B, NOISE_DIM, device=DEVICE)
            x_fake = G(z)
            d_fake = D_for_G(x_fake)
            l_g    = F.binary_cross_entropy_with_logits(d_fake, torch.ones_like(d_fake))
            opt_g.zero_grad()
            l_g.backward()
            opt_g.step()
            g_ep.append(l_g.item())

        losses_g.append(np.mean(g_ep))
        losses_d.append(np.mean(d_ep))

    if privacy_engine is not None:
        actual_epsilon = privacy_engine.get_epsilon(delta=DELTA)

    elapsed = (time.time() - t0) / 60
    print(f"  [{label}] ε_real={actual_epsilon:.2f}  tiempo={elapsed:.1f} min  "
          f"L_G={losses_g[-1]:.4f}  L_D={losses_d[-1]:.4f}")

    return {
        "generator":        G,
        "losses_g":         losses_g,
        "losses_d":         losses_d,
        "actual_epsilon":   actual_epsilon,
        "noise_multiplier": noise_multiplier,
    }

## 4. Epsilon sweep

Se entrenan 4 modelos en secuencia: ε=∞ (sin DP), ε=10, ε=5, ε=1.
Los modelos con menor ε reciben más ruido en sus gradientes y producen
datos sintéticos de menor fidelidad pero con mayor garantía de privacidad.

In [ ]:
results = {}

for eps_target, eps_label in zip(EPSILON_TARGETS, EPSILON_LABELS):
    print(f"\nEntrenando {eps_label}...")
    results[eps_label] = train_dp_gan(
        X_data=X,
        target_epsilon=eps_target,
        n_epochs=N_EPOCHS,
        batch_size=BATCH,
        label=eps_label,
    )

print("\n=== Resumen del epsilon sweep ===")
print(f"{'Config':<8} {'ε target':>10} {'ε real':>10} {'σ':>8} {'L_G final':>10} {'L_D final':>10}")
print("-" * 58)
for (eps_target, eps_label) in zip(EPSILON_TARGETS, EPSILON_LABELS):
    r = results[eps_label]
    tgt = "∞" if eps_target is None else str(eps_target)
    print(f"{eps_label:<8} {tgt:>10} {r['actual_epsilon']:>10.2f} "
          f"{r['noise_multiplier']:>8.4f} {r['losses_g'][-1]:>10.4f} {r['losses_d'][-1]:>10.4f}")

In [ ]:
# Curvas de pérdida comparadas
colors = {"ε=∞": "steelblue", "ε=10": "seagreen", "ε=5": "orange", "ε=1": "tomato"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for eps_label, r in results.items():
    c = colors[eps_label]
    axes[0].plot(r["losses_g"], color=c, linewidth=1.2, label=eps_label, alpha=0.85)
    axes[1].plot(r["losses_d"], color=c, linewidth=1.2, label=eps_label, alpha=0.85)

axes[0].set_title("Generator Loss")
axes[0].set_xlabel("Época")
axes[0].legend()
axes[1].set_title("Discriminator Loss")
axes[1].set_xlabel("Época")
axes[1].legend()

fig.suptitle("Convergencia por configuración de privacidad", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "dp_ctgan_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/dp_ctgan_loss_curves.png")

## 5. Generación de muestras sintéticas

In [ ]:
N_SAMPLES   = len(tab)
synth_dfs   = {}

for eps_label, r in results.items():
    G = r["generator"]
    G.eval()
    with torch.no_grad():
        chunks = []
        for start in range(0, N_SAMPLES, BATCH):
            end = min(start + BATCH, N_SAMPLES)
            z   = torch.randn(end - start, NOISE_DIM, device=DEVICE)
            chunks.append(G(z).cpu().numpy())
        X_synth = np.concatenate(chunks, axis=0)

    synth_df = preprocessor.inverse_transform(X_synth)
    synth_dfs[eps_label] = synth_df

    # Guardar parquet
    eps_str = eps_label.replace("=", "").replace("∞", "inf")
    fname   = SYNTHETIC / f"dp_ctgan_{eps_str}_samples.parquet"
    synth_df.to_parquet(fname, index=False)
    print(f"  [{eps_label}] Guardado: {fname.name}  shape={synth_df.shape}")

# Guardar checkpoints de los generadores
for eps_label, r in results.items():
    eps_str = eps_label.replace("=", "").replace("∞", "inf")
    torch.save(
        {"state_dict": r["generator"].state_dict(),
         "actual_epsilon": r["actual_epsilon"],
         "noise_multiplier": r["noise_multiplier"]},
        MODELS / f"dp_ctgan_{eps_str}.pt",
    )
print("\nCheckpoints guardados en models/")

## 6. Tradeoff Privacidad – Utilidad

### 6.1 Distribuciones marginales por configuración

In [ ]:
n_cols_plot = min(4, len(KEY_COLS))
fig, axes = plt.subplots(len(EPSILON_LABELS), n_cols_plot,
                          figsize=(n_cols_plot * 4, len(EPSILON_LABELS) * 3))

for row, (eps_label, synth_df) in enumerate(synth_dfs.items()):
    for col, var in enumerate(KEY_COLS[:n_cols_plot]):
        ax = axes[row][col]
        if var in tab.columns:
            ax.hist(tab[var].dropna(), bins=50, alpha=0.5, density=True,
                    color="gray", label="Real")
        if var in synth_df.columns:
            ax.hist(synth_df[var].dropna(), bins=50, alpha=0.6, density=True,
                    color=colors[eps_label], label=eps_label)
        if row == 0:
            ax.set_title(var, fontsize=9)
        if col == 0:
            ax.set_ylabel(eps_label, fontsize=9, rotation=0, labelpad=45)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6)

fig.suptitle("Distribuciones marginales por nivel de privacidad (ε)", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "dp_ctgan_marginals_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/dp_ctgan_marginals_grid.png")

### 6.2 Métrica de utilidad — MAE normalizado en estadísticos marginales

Para cada configuración de ε se calcula el **Mean Absolute Error normalizado**
entre la media de cada variable clave en el dataset real y el sintético,
dividido por la desviación estándar real:

$$\text{MAE}_\text{norm} = \frac{1}{|V|} \sum_{v \in V} \frac{|\mu_v^\text{real} - \mu_v^\text{synth}|}{\sigma_v^\text{real}}$$

Un valor de 0 indica fidelidad perfecta; valores > 0.10 indican discrepancia clínica relevante.

In [ ]:
def marginal_mae_norm(real_df, synth_df, cols):
    errors = []
    for col in cols:
        if col not in real_df.columns or col not in synth_df.columns:
            continue
        sigma = real_df[col].std()
        if sigma == 0:
            continue
        errors.append(abs(real_df[col].mean() - synth_df[col].mean()) / sigma)
    return np.mean(errors) if errors else np.nan


utility_rows = []
print(f"{'Config':<8} {'ε real':>8} {'MAE norm (↓ mejor)':>20}")
print("-" * 42)
for eps_label, synth_df in synth_dfs.items():
    eps_real = results[eps_label]["actual_epsilon"]
    mae      = marginal_mae_norm(tab, synth_df, KEY_COLS)
    utility_rows.append({"label": eps_label, "epsilon": eps_real, "mae_norm": mae})
    print(f"{eps_label:<8} {eps_real:>8.2f} {mae:>20.4f}")

utility_df = pd.DataFrame(utility_rows)
utility_df.to_csv(REPORTS / "dp_ctgan_utility_tradeoff.csv", index=False)
print("\nGuardado: reports/dp_ctgan_utility_tradeoff.csv")

In [ ]:
# Figura principal del TFG: curva privacidad–utilidad
fig, ax = plt.subplots(figsize=(7, 5))

# Separar el punto ε=∞ para plotear como referencia
finite_rows = utility_df[utility_df["epsilon"] < 1e6].sort_values("epsilon")
inf_row     = utility_df[utility_df["epsilon"] >= 1e6]

# Línea de referencia (sin DP)
if not inf_row.empty:
    ax.axhline(inf_row["mae_norm"].values[0], color="gray", linestyle="--",
               linewidth=1.5, label="Sin DP (ε=∞)")
    ax.text(finite_rows["epsilon"].min() * 1.05, inf_row["mae_norm"].values[0] * 1.01,
            "ε=∞ (sin DP)", fontsize=9, color="gray")

# Curva DP
if not finite_rows.empty:
    ax.plot(finite_rows["epsilon"], finite_rows["mae_norm"],
            marker="o", markersize=8, linewidth=2, color="steelblue",
            label="DP-CTGAN")
    for _, row in finite_rows.iterrows():
        ax.annotate(f" ε={row['epsilon']:.1f}",
                    (row["epsilon"], row["mae_norm"]),
                    fontsize=8, va="bottom")

ax.set_xlabel("Presupuesto de privacidad ε (↓ más privado)", fontsize=11)
ax.set_ylabel("MAE normalizado (↓ mejor utilidad)", fontsize=11)
ax.set_title("Tradeoff Privacidad–Utilidad — DP-CTGAN sobre MIMIC-III", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(REPORTS / "dp_ctgan_privacy_utility_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/dp_ctgan_privacy_utility_curve.png")

## 7. Resumen

In [ ]:
print("=" * 65)
print("  RESUMEN — Notebook 07")
print("=" * 65)
print(f"  {'Configuración':<12} {'ε real':>8} {'σ':>8} {'MAE norm':>12}")
print("  " + "-" * 44)
for (eps_label, r), row in zip(results.items(), utility_rows):
    print(f"  {eps_label:<12} {r['actual_epsilon']:>8.2f} "
          f"{r['noise_multiplier']:>8.4f} {row['mae_norm']:>12.4f}")
print("=" * 65)
print(f"  delta (δ):         {DELTA}")
print(f"  max_grad_norm (C): {MAX_GRAD_NORM}")
print(f"  N_EPOCHS:          {N_EPOCHS}")
print(f"  batch_size:        {BATCH}")
print("=" * 65)
print("Fase 2 completada. Listos para Fase 3 — evaluación de fidelidad.")